In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import re
import seaborn as sns
from analyze import analysis
from headcast.headcast_funcs import csv_path_list, get_all_labels, dir_data, print_dict_tree, transform_dict
from plot import gen_figure
from pprint import pprint
from glob import glob
import contextlib
import os

In [ ]:
label_map = {0:'straight', 1: 'straight', 2:'left cast', 3:'left shallow turn',
                4:'left sharp turn', 5:'right cast', 6:'right shallow turn', 7:'right sharp turn'}
tanish_label_map = {0:'straight', 1: '', 2:'left cast',
                3:'left turn', 4:'', 5:'right cast', 6:'right turn'}

# Define paths to label dirs

In [ ]:
nitesh_hl = '/Users/hind/Documents/UCSB/Neuroscience/headcasting_project/Data_from_NAS/headlocked_center'
tanish_hl = '/Users/hind/Documents/UCSB/Neuroscience/headcasting_project/Tanish_preds/all_data/all-data-0.50/Merged/headlocked'
tanish_hl75 = '/Users/hind/Documents/UCSB/Neuroscience/headcasting_project/Tanish_preds/all_data/all-data-0.75/Merged/headlocked'
peris = csv_path_list(nitesh_hl)
q50 = csv_path_list(tanish_hl, invert=True)
q75 = csv_path_list(tanish_hl75, invert=True)


In [ ]:
print(len(peris))
print(len(q50))
print(len(q75))

In [ ]:
thl = get_all_labels(q50, col=2)
print(np.unique(thl))

thl = get_all_labels(q50, col=[3,4])
print(np.unique(thl))

In [ ]:
bins=[-0.45, 0.45, 0.55, 1.45, 1.55, 2.45, 2.55, 3.45, 3.55,
                                            4.45, 4.55, 5.45, 5.55, 6.45, 6.55, 7.45, 7.55, 8.45, 8.55, 9.45, 9.55, 10.45, 10.55, 11.45, 11.55]
plt.hist(get_all_labels(q50, col=[3,4]), bins=bins, alpha=0.5)
plt.xticks(np.arange(0, 12, 1))

In [ ]:
bin_trio = [-1.45, -0.55, -0.45, 0.45, 0.55, 1.45]

plt.hist(get_all_labels(csv_path_list(tanish_hl), col=5), bins=bin_trio, alpha=0.5, label='supervised ML')
plt.hist(get_all_labels(csv_path_list(nitesh_hl), col=5), bins=bin_trio, alpha=0.5, label='peristalsis')
plt.xticks([-1, 0, 1], ['reject', 'N/A', 'accept'])
plt.legend()
plt.show()

In [ ]:
bins = [-0.45, 0.45, 0.55, 1.45, 1.55, 2.45, 2.55, 3.45, 3.55, 4.45, 4.55, 5.45, 5.55, 6.45]
q5 = get_all_labels(q50)
perit = get_all_labels(peris)

plt.hist(q5, bins=bins, alpha=0.5, label='quantile 50', density=True)
plt.hist(perit, bins=bins, alpha=0.5, label= 'peristalsis', density=True)
# plt.hist(get_all_labels(li_peristalsis_t), bins=[-0.55, 0.45, 0.55, 1.45, 1.55, 2.45, 2.55, 3.45, 3.55,
#                                                4.45, 4.55, 5.45, 5.55, 6.45, 6.55, 7.45], alpha=0.6, label=label_map.values())
plt.xticks([ 0, 2, 3, 5, 6], [ 'straight', 'left\ncast', 'left\nturn', 'right\ncast', 'right\nturn'])
plt.legend()
# plt.title(f'{condition}')


In [ ]:
all_peristalsis = get_all_labels(peris)
all_q50 = get_all_labels(q50)

## generate ethograms

In [ ]:
basedir = '/Users/hind/Documents/UCSB/Neuroscience/headcasting_project/Tanish_preds/all_data/all-data-0.50/headlocked_figs'
cond = 'headlocked'
for i in range(1, len(q50)):
    # print(np.unique(get_all_labels(c_q50[i], col=2)), np.unique(get_all_labels(c_q625[i], col=2)), np.unique(get_all_labels(c_q75[i], col=2)), np.unique(get_all_labels(peri[i], col=2)))
    fig = gen_figure(q50[i], peris[i], use_cols=3, return_fig=True)
    plt.savefig(os.path.join(basedir, f'{cond}_{i}_casts.png'), dpi = 150)
    plt.close()

    fig = gen_figure(q50[i], peris[i], use_cols=4, return_fig=True)
    plt.savefig(os.path.join(basedir, f'{cond}_{i}_turns.png'), dpi = 150)
    plt.close()

    fig = gen_figure(q50[i], peris[i], use_cols=2, return_fig=True)
    plt.savefig(os.path.join(basedir, f'{cond}_{i}_col2.png'), dpi = 150)
    plt.close()

    fig = gen_figure(q50[i], peris[i], use_cols=[3,4], return_fig=True)
    plt.savefig(os.path.join(basedir, f'{cond}_{i}_merged.png'), dpi = 150)
    plt.close()

    fig = gen_figure(q50[i], peris[i], use_cols=5, return_fig=True)
    plt.savefig(os.path.join(basedir, f'{cond}_{i}_accept.png'), dpi = 150)
    plt.close()

In [ ]:
def get_analysis_data(*csv_path_lists, col=2):
    all_info = {}
    if len(csv_path_lists) < 2:
        raise ValueError("At least two lists are required")
    min_len = min(len(lst) for lst in csv_path_lists)
    for i in range(min_len):
        # Unpack the i-th element from each list
        csvs = [lst[i] for lst in csv_path_lists]
        print(csvs)
        # Pass all csvs to analysis using *
        info = analysis(*csvs, col=col)
        for key in info:
            if key not in all_info and '.csv' in key:
                all_info[key] = info[key]
            if key == 'comparison_results':
                all_info[i+1] = info[key]

    transformed_info = transform_dict(all_info)
    return transformed_info

## compute metrics

In [ ]:
with open(os.devnull, 'w') as f, contextlib.redirect_stdout(f):
    all_info_col2 = get_analysis_data(peris, q50)
    cast_info = get_analysis_data(peris, q50, col=3)
    turn_info = get_analysis_data(peris, q50, col=4)

In [ ]:
print(print_dict_tree(all_info[1]))

In [ ]:
def get_match_score(info):
    best_match = []
    alignment_score = []
    for key in info.keys():
        best_match.append(info[key]['comparison']['best_match'])
        alignment_score.append(info[key]['comparison']['alignment_score'])
    return np.array(best_match), np.array(alignment_score)

In [ ]:
get_match_score(all_info)

In [ ]:
def get_confusion_matrix(info):
    cm = []
    for key in info.keys():
        cmatrix = info[key].get('conf_matrix', None)
        if cmatrix is not None:
            cm.append(cmatrix)
    return cm

def condition_cm(dictionary):
    cm = get_confusion_matrix(dictionary)
    final_cm = np.zeros((5,5))
    for i in range(len(cm)):
        final_cm += cm[i]
    final_cm = final_cm.astype(int)
    return final_cm

In [ ]:
cm = condition_cm(turn_info)

In [ ]:
print(np.array(cm))

In [ ]:
cast_cm = cm[:4, :4]
cast_cm = np.delete(cast_cm, 2, axis=1)
cast_cm = np.delete(cast_cm, 2, axis=0)

In [ ]:
turn_cm = np.delete(cm, [1,3], axis=1)
turn_cm = np.delete(turn_cm, [1,3], axis=0)

In [ ]:
print(turn_cm)

In [ ]:
print(cast_cm)

### full confusion matrix

In [ ]:
annot_labels = ['straight', 'left cast', 'left turn', 'right cast', 'right turn']
norm_cm = final_cm
norm_cm = final_cm / final_cm.sum(axis=0, keepdims=True)
fig, ax = plt.subplots()
sns.heatmap(norm_cm, annot=True, fmt='.2f', xticklabels=annot_labels, yticklabels=annot_labels, cmap='Blues', ax=ax)
ax.set_xlabel('Tanish')
ax.xaxis.set_label_position('top')
ax.xaxis.tick_top()
plt.ylabel('Nitesh')
plt.show(block=False)

### cast CM

In [ ]:
annot_labels = ['straight', 'left cast', 'right cast']
# norm_cm = cast_cm
norm_cm = cast_cm / cast_cm.sum(axis=0, keepdims=True)
fig, ax = plt.subplots()
sns.heatmap(norm_cm, annot=True, fmt='.2f', xticklabels=annot_labels, yticklabels=annot_labels, cmap='Blues', ax=ax)
ax.set_xlabel('Tanish')
ax.xaxis.set_label_position('top')
ax.xaxis.tick_top()
plt.ylabel('Nitesh')
plt.show(block=False)

In [ ]:
tcast = get_all_labels(q50, col=3)
ncast = get_all_labels(peris, col=3)
plt.hist(tcast, bins=bins[:-2], alpha=0.5, label='supervised ML', density=True)
plt.hist(ncast, bins=bins[:-2], alpha=0.5, label= 'peristalsis', density=True)
plt.xticks([ 0, 2, 5], [ 'straight', 'left\ncast', 'right\ncast'])
plt.legend()

In [ ]:
tturn = get_all_labels(q50, col=4)
nturn = get_all_labels(peris, col=4)
plt.hist(tturn, bins=bins, alpha=0.5, label='supervised ML', density=True)
plt.hist(nturn, bins=bins, alpha=0.5, label= 'peristalsis', density=True)
plt.xticks([ 0, 3, 6], [ 'straight', 'left\nturn', 'right\nturn'])
plt.legend()

### turn CM

In [ ]:
annot_labels = ['straight', 'left turn', 'right turn']
# norm_cm = cast_cm
norm_cm = turn_cm / turn_cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots()
sns.heatmap(norm_cm, annot=True, fmt='.2f', xticklabels=annot_labels, yticklabels=annot_labels, cmap='Blues', ax=ax)
ax.set_xlabel('Tanish')
ax.xaxis.set_label_position('top')
ax.xaxis.tick_top()
plt.ylabel('Nitesh')
plt.show(block=False)

In [ ]:
newdict = transform_dict(all_info)

In [ ]:
print(newdict[1].keys())

In [ ]:
print_dict_tree(newdict[1])


In [ ]:
accuracy_50 = []
accuracy_625 = []
accuracy_75 = []
for key in all_info.keys():
    if '.csv' in str(key):
        # if '625' in key:
        #     accuracy_625.append(all_info[key]['accuracy'])
        if 'invert' in key:
            accuracy_50.append(all_info[key]['accuracy'])
        # elif '75' in key:
        #     accuracy_75.append(all_info[key]['accuracy'])

In [ ]:
print(accuracy_50)

In [ ]:
condition = 'headlocked'
plt.boxplot([accuracy_50, accuracy_625, accuracy_75], tick_labels=['50', '62.5', '75'], showfliers=False)
plt.ylabel('Match Accuracy')
plt.xlabel('Quantile Threshold')
plt.title(f'Accuracy of label match with peristalsis - {condition}')
plt.show()

In [ ]:
bm50, as50 = get_match_score(all_info, '50')
bm75, as75 = get_match_score(all_info, '75')
bm625, as625 = get_match_score(all_info, '625')

In [ ]:
plt.boxplot([bm50, bm625, bm75], tick_labels=['50', '62.5', '75'], showfliers=False)
plt.ylim(0, 1)
plt.ylabel('Best Match Score')
plt.xlabel('Quantile Threshold')
plt.title(f'Best Match Score with peristalsis labels - {condition}')
plt.show()

In [ ]:
plt.boxplot([as50, as625, as75], tick_labels=['50', '62.5', '75'], showfliers=False)
plt.ylabel('Alignment Score')
plt.xlabel('Quantile Threshold')
plt.title(f'Alignment Score with peristalsis labels - {condition}')
plt.show()

In [ ]:
seg_lengths_625 = {0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: []}
seg_lengths_75 = {0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: []}
seg_lengths_50 = {0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: []}
seg_lengths_peristalsis = {0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: []}

counts_625 = {0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: []}
counts_75 = {0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: []}
counts_50 = {0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: []}
counts_peristalsis = {0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: []}

for k in all_info.keys():
    if 'anish' in str(k):
        for key in all_info[k]['seg_lengths'].keys():
        #     if 'quantile_625' in k and 'quantile_625' in key:
        #         for n in all_info[k]['seg_lengths'][key].keys():
        #             seg_lengths_625[n].extend(all_info[k]['seg_lengths'][key][n])
        #             counts_625[n].append(all_info[k]['counts'][n])
            # elif 'quantile_75' in k and 'quantile_75' in key:
            #     for n in all_info[k]['seg_lengths'][key].keys():
            #         seg_lengths_75[n].extend(all_info[k]['seg_lengths'][key][n])
            #         counts_50[n].append(all_info[k]['counts'][n])
            if 'anish' in k and 'invert' in key:
                for n in all_info[k]['seg_lengths'][key].keys():
                    seg_lengths_50[n].extend(all_info[k]['seg_lengths'][key][n])
                    counts_50[n].append(all_info[k]['counts'][n])
    elif 'NAS' in str(k):
        if 'NAS' in key:
            for n in all_info[k]['seg_lengths'][key].keys():
                seg_lengths_peristalsis[n].extend(all_info[k]['seg_lengths'][key][n])
                counts_peristalsis[n].append(all_info[k]['counts'][n])

In [ ]:
print(counts_50)

In [ ]:
condition = 'headlocked'



plt.figure(figsize=(10, 6))
# box1 = plt.boxplot([counts_75[key] for key in counts_75.keys()], 
#                    positions=[0, 1, 2, 3, 4, 5, 6], widths=0.15, patch_artist=True, 
#                    showfliers=False, label='75 Quantile')
# for patch in box1['boxes']:
#     patch.set_facecolor('#ADD8E6')  # Set color for the first boxplot

# # Create the second boxplot and set its color
# box2 = plt.boxplot([counts_625[key] for key in counts_625.keys()], 
#                    positions=[0.2, 1.2, 2.2, 3.2, 4.2, 5.2, 6.2], widths=0.15, 
#                    patch_artist=True, showfliers=False, label='62.5 Quantile')
# for patch in box2['boxes']:
#     patch.set_facecolor('#90EE90')  # Set color for the second boxplot

# Create the third boxplot and set its color
box3 = plt.boxplot([counts_50[key] for key in counts_50.keys()], 
                   positions=[0.4, 1.4, 2.4, 3.4, 4.4, 5.4, 6.4], widths=0.15, 
                   patch_artist=True, showfliers=False, label='50 Quantile')
for patch in box3['boxes']:
    patch.set_facecolor('#FFB6C1')  # Set color for the third boxplot

box4 = plt.boxplot([counts_peristalsis[key] for key in counts_peristalsis.keys()],
                   positions=[0.6, 1.6, 2.6, 3.6, 4.6, 5.6, 6.6], widths=0.15,
                   patch_artist=True, showfliers=False, label = 'Peristalsis')
for patch in box4['boxes']:
    patch.set_facecolor('#FFD700')

plt.xticks([0.3, 2.3, 3.3, 5.3, 6.3], 
           labels = ['straight', 'left cast', 'left turn', 'right cast', 'right turn'])
plt.legend()
plt.title(f'Number of label segments - {condition}')
plt.ylabel('Number of segments per trajectory')
plt.show()


In [ ]:
print(len(np.array(counts_peristalsis[0])))

In [ ]:
seg_lengths_peristalsis.keys()

In [ ]:
label = 0
# behavior = 'Straight'

plt.figure(figsize=(10, 6))
plt.title(f'Left Increase - {condition} lengths')
plt.hist(seg_lengths_peristalsis[label], bins=30, alpha = 0.9, label='peristalsis', density=True)
plt.hist(seg_lengths_50[label], bins=30, alpha = 0.5, label='supervised 50 quantile', density=True)
# plt.hist(seg_lengths_625[label], bins=35, alpha = 0.5, label='supervised 62.5 quantile', density=True)
# plt.hist(seg_lengths_75[label], bins=35, alpha = 0.5, label='supervised 75 quantile', density=True)
plt.yscale('log')
# plt.xlim(-1, 800)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

box1 = plt.boxplot([seg_lengths_peristalsis[label] for label in seg_lengths_peristalsis.keys()], 
            positions=[0.6, 1.6, 2.6, 3.6, 4.6, 5.6, 6.6], widths=0.15,
                   patch_artist=True, showfliers=False, label = 'Peristalsis')
for patch in box1['boxes']:
    patch.set_facecolor('#FFD700')

box2 = plt.boxplot([seg_lengths_75[key] for key in seg_lengths_75.keys()], 
                   positions=[0, 1, 2, 3, 4, 5, 6], widths=0.15, patch_artist=True, 
                   showfliers=False, label='75 Quantile')
for patch in box2['boxes']:
    patch.set_facecolor('#ADD8E6')  # Set color for the first boxplot

# Create the second boxplot and set its color
box3 = plt.boxplot([seg_lengths_625[key] for key in seg_lengths_625.keys()], 
                   positions=[0.2, 1.2, 2.2, 3.2, 4.2, 5.2, 6.2], widths=0.15, 
                   patch_artist=True, showfliers=False, label='62.5 Quantile')
for patch in box3['boxes']:
    patch.set_facecolor('#90EE90')  # Set color for the second boxplot

# Create the third boxplot and set its color
box4 = plt.boxplot([seg_lengths_50[key] for key in seg_lengths_50.keys()], 
                   positions=[0.4, 1.4, 2.4, 3.4, 4.4, 5.4, 6.4], widths=0.15, 
                   patch_artist=True, showfliers=False, label='50 Quantile')
for patch in box4['boxes']:
    patch.set_facecolor('#FFB6C1')  # Set color for the third boxplot



plt.xticks([0.3, 2.3, 3.3, 5.3, 6.3], 
           labels = ['straight', 'left cast', 'left turn', 'right cast', 'right turn'])
plt.legend()
plt.title(f'Length of label segments - {condition}')
plt.ylabel('Length of behavioral segment')
plt.show()

# plt.figure(figsize=(10, 6))
# plt.title(f'Left Increase - {behavior} lengths')
# plt.hist(seg_lengths_peristalsis[label], bins=50, alpha = 0.9, label='peristalsis', density=True)
# plt.hist(seg_lengths_50[label], bins=40, alpha = 0.5, label='supervised 50 quantile', density=True)
# plt.hist(seg_lengths_625[label], bins=40, alpha = 0.5, label='supervised 62.5 quantile', density=True)
# plt.hist(seg_lengths_75[label], bins=40, alpha = 0.5, label='supervised 75 quantile', density=True)

In [ ]:
for k in seg_lengths_peristalsis.keys():
    print(k, np.nanmean(seg_lengths_peristalsis[k]), np.nanstd(seg_lengths_peristalsis[k]), np.nanmedian(seg_lengths_peristalsis[k]))

In [ ]:
for k in seg_lengths_625.keys():
    print(k, np.nanmean(seg_lengths_625[k]), np.nanstd(seg_lengths_625[k]), np.nanmedian(seg_lengths_625[k]))

In [ ]:
from glob import glob

In [ ]:
files = glob('/Users/hind/Documents/UCSB/Neuroscience/headcasting project/Filtered Supervised vs Peristalsis/Persistalsis/right_decrease/*_pc.csv')

In [ ]:
for i in range(len(files)):
    f = pd.read_csv(files[i], header=None)
    # print(f[2].values)
    print(np.unique(f[2].values))
    print(np.unique(f[3].values))
    print(np.unique(f[4].values))
    x = np.where(f[3].values != 0)[0]
    y = np.where(f[4].values != 0)[0]
    z = np.intersect1d(x, y)
    print(len(z))
    print(z)
    if len(z) > 0:
        print(os.path.basename(files[i]))

    print('------------')

In [ ]:
def csv_list(path):
    csv_files = [os.path.join(path, file) for file in os.listdir(path) if file.endswith('.csv')]
    file_list = [f for f in csv_files if 'locked' in os.path.basename(f)]

    file_list.sort(key=extract_number)

    return file_list

In [ ]:
taillocked  = csv_list(q50_path)

In [ ]:
print(taillocked)

In [ ]:
def merge_columns_in_csv(file_path):
    """
    Reads a CSV file with 3 columns, processes columns 1 and 2, and writes the updated data back to the same file.
    If column 2 has a zero and column 1 has a nonzero integer, the nonzero value is moved to column 2, and column 1 is set to 0.
    
    Args:
        file_path (str): Path to the CSV file.
    """
    # Read the CSV file
    df = pd.read_csv(file_path, header=None)
    
    # Ensure columns 1 and 2 are integers
    df[1] = df[1].astype(int)
    df[2] = df[2].astype(int)
    
    # Process the columns
    mask = (df[2] == 0) & (df[1] != 0)
    df.loc[mask, 2] = df.loc[mask, 1]  # Move nonzero values from column 1 to column 2
    df.loc[mask, 0] = 0  # Set column 1 to 0 for those rows
    
    # Write the updated data back to the same file
    df.to_csv(file_path, header=False, index=False)

In [ ]:
for i in range(len(taillocked)):
    merge_columns_in_csv(taillocked[i])
    print(taillocked[i])
    print(np.unique(get_all_labels(taillocked[i], col=2)))
    print('------------------')